# 1 · System preparation

This is the first stage of the pipeline. It acquires the host and guest
structures, curates them, and geometry-optimizes every system at a single
consistent level of theory (**GFN1-xTB/D3(BJ)**, AMS DFTB engine) so that all
binding energies computed later share one common energy reference.

It implements methodology **§1–§3**, documented in
[`system-preparation.md`](../system-preparation/system-preparation.md).

> **Pipeline order:** `system-preparation` → `structural-characterization` → `modeling-and-simulation` → `energetic-data-analysis` → `selectivity-analysis`.

**Prerequisites**
- The [`mof-guest-toolkit`](https://github.com/adricu12/mof-guest-toolkit) package (`pip install mof-guest-toolkit`), which provides `get_3d_structure`, `read_structure`, `gen_inp_file`, `parse_ams_output`, and `get_final_structure`.
- [AMS](https://www.scm.com/) for the GFN1-xTB calculations (run separately on a workstation or HPC — see §3). An internet connection is needed for the PubChem download.

**What is committed here.** The *curated* and *geometry-optimized* structures
(`thesis/011_HOSTS/`, `thesis/012_GUESTS/`) are committed, so the downstream
notebooks run without re-running any quantum chemistry. The bulky AMS input/output
files are regenerated by the cells below and are not committed (see
[`thesis/README.md`](../thesis/README.md)). Cells are committed without execution
outputs; re-run the notebook top to bottom to reproduce them.

The setup cell below imports the toolkit and defines the three base paths used
throughout this notebook.

In [ ]:
from pathlib import Path

from mof_toolkit import (
    get_3d_structure,    # download a PubChem 3D conformer
    read_structure,      # read an xyz/cif into (symbols, coords, lattice, fractional)
    gen_inp_file,        # fill an AMS/ORCA template from a structure
    parse_ams_output,    # parse an AMS .out file
    get_final_structure,  # extract the optimized geometry from a parsed output
)

# Three base paths resolve every file relative to the repository root, so the
# notebook runs unchanged after cloning as long as the layout is intact.
DATA      = Path("../data")        # reference inputs (guest list, simulation seeds)
THESIS    = Path("../thesis")      # numbered pipeline stages (inputs, intermediates, results)
TEMPLATES = Path("./templates")    # hand-written calculation input templates

## 1. Structure acquisition

### 1.1 Host frameworks (manual)
The four porphyrinic Zr-MOF crystal structures were downloaded from the CCDC
(deposition numbers in
[`system-preparation.md` §1.1](../system-preparation/system-preparation.md#11-host-structures))
and are not redistributed here. Their curated versions are committed under
`thesis/011_HOSTS/curated/`.

### 1.2 Guest molecules — PubChem download
Each guest's 3D conformer is fetched from PubChem by CID via
`get_3d_structure`. The CID list is [`data/guests.csv`](../data/guests.csv);
files are named by the guest abbreviation (e.g. `AC.xyz` for acetaminophen).

In [ ]:
import pandas as pd

# The guest list (name, abbreviation, PubChem CID, type) is the single source
# of truth for the 37 guests; see ../system-preparation/system-preparation.md §1.2.
guests = pd.read_csv(DATA / "guests.csv")

pristine_dir = THESIS / "012_GUESTS" / "pristine_structures"
pristine_dir.mkdir(parents=True, exist_ok=True)

# Download each guest's PubChem 3D conformer, saved as <abbreviation>.xyz.
for _, row in guests.iterrows():
    print(f"{row['Name']:<35} CID {row['CID']!s:>9} -> {row['Abbreviation']}.xyz")
    get_3d_structure(
        row["CID"],
        formats=["xyz"],
        output_stem=str(pristine_dir / row["Abbreviation"]),
    )

## 2. Curation (manual)

As-deposited host CIFs are curated by removing lattice and coordinated solvent
molecules and verifying the μ₃-OH hydrogens on the Zr₆ SBU; PCN-225 is reduced
to its primitive cell. Guest conformers are checked for the correct protonation
state at neutral pH. This is a manual inspection step (see
[`system-preparation.md` §2](../system-preparation/system-preparation.md#2-structure-curation-and-preparation));
its committed outputs are `thesis/011_HOSTS/curated/` and `thesis/012_GUESTS/curated/`.

## 3. Geometry optimization — input generation

Every curated structure is optimized at GFN1-xTB/D3(BJ) with AMS. The cell below
fills the [`geo-opt_ams.run`](./templates/geo-opt_ams.run) template with each
structure's coordinates — and lattice vectors, for the periodic hosts — writing
one `.run` file per system into its own folder.

Hosts (`.cif`, periodic) and guests (`.xyz`, molecular) are handled by the **same
helper**; only the file extension differs. `read_structure` returns
`(symbols, coords, lattice, fractional)`, which unpacks straight into
`gen_inp_file` — for `.xyz` files `lattice` is `None`, so the template's
`Lattice` block is dropped automatically.

In [ ]:
def generate_geo_opt_inputs(structure_dir, output_dir, ext):
    """Write one AMS geo-opt `.run` file per structure file in `structure_dir`.

    Each structure gets its own `<name>/` folder holding `<name>_geo-opt.run`,
    ready for individual HPC submission.
    """
    structure_dir, output_dir = Path(structure_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    template = str(TEMPLATES / "geo-opt_ams.run")

    for path in sorted(structure_dir.glob(f"*.{ext}")):
        name = path.stem
        run_dir = output_dir / name
        run_dir.mkdir(exist_ok=True)
        gen_inp_file(
            template,
            *read_structure(str(path), coord_type="cart"),
            name=f"{name}_geo-opt",
            output=str(run_dir / f"{name}_geo-opt.run"),
        )
    print(f"Generated {sum(1 for _ in output_dir.iterdir())} input folders in {output_dir}/")


# Guests (molecular .xyz) and hosts (periodic .cif) share the same routine.
generate_geo_opt_inputs(THESIS / "012_GUESTS/curated", THESIS / "012_GUESTS/geo_opt", ext="xyz")
generate_geo_opt_inputs(THESIS / "011_HOSTS/curated",  THESIS / "011_HOSTS/geo_opt",  ext="cif")

### Running the calculations

The generated `.run` files are GFN1-xTB geometry optimizations — light enough for
a workstation, but most conveniently submitted to an HPC cluster with SLURM, one
job per structure. A representative submission script is in
[`templates/submit-hpc.sh`](./templates/submit-hpc.sh):

```bash
#!/bin/bash
#SBATCH --job-name=<job>
#SBATCH --nodes=1
#SBATCH --ntasks=<n>
#SBATCH --time=hh:mm:ss
#SBATCH --mem=<m>G
#SBATCH --output=%x_%j.out
#SBATCH --error=%x_%j.err

module purge
module add <AMS module>

"$AMSBIN/ams" < <job>_geo-opt.run > <job>_geo-opt.out
```

Each job writes `<name>_geo-opt.out` into its folder. The next cell parses every
output and extracts the optimized geometry — CIF for the periodic hosts, XYZ for
the molecular guests.

In [ ]:
def collect_optimized(geo_opt_dir, final_dir, name_fn):
    """Extract the optimized structure from each AMS `.out` into `final_dir`.

    `name_fn(folder_name)` builds the output filename; its extension (`.cif` /
    `.xyz`) determines the format `get_final_structure` writes.
    """
    geo_opt_dir, final_dir = Path(geo_opt_dir), Path(final_dir)
    final_dir.mkdir(parents=True, exist_ok=True)

    for folder in sorted(p for p in geo_opt_dir.iterdir() if p.is_dir()):
        out_file = folder / f"{folder.name}_geo-opt.out"
        if not out_file.is_file():
            continue
        get_final_structure(
            parse_ams_output(str(out_file)),
            output=str(final_dir / name_fn(folder.name)),
        )
        print(f"  {folder.name} -> {name_fn(folder.name)}")


# Guests keep their abbreviation; hosts are tagged with the method and task.
collect_optimized(
    THESIS / "012_GUESTS/geo_opt", THESIS / "012_GUESTS/final_structures",
    name_fn=lambda n: f"{n}.xyz",
)
collect_optimized(
    THESIS / "011_HOSTS/geo_opt", THESIS / "011_HOSTS/final_structures",
    name_fn=lambda n: f"{n.replace('_curated', '')}_dftb_geo-opt.cif",
)

---

### Reference single-point energies

A single-point energy at the same level of theory (GFN1-xTB/D3(BJ), AMS template
with `Task SinglePoint`) is then run on each optimized host and guest. These
provide the reference terms $E_{\text{host}}$ and $E_{\text{guest}}$ in the
binding-energy expression and are reused in
[`energetic-data-analysis`](./energetic-data-analysis.ipynb) (§6).

**Next:** [`structural-characterization.ipynb`](./structural-characterization.ipynb)
— host pore geometry (Zeo++) and guest molecular descriptors (RDKit).